# Anti-Hallucination Guardrails

## Why Anti-Hallucination is Critical

AI hallucinations occur when language models generate information that is:
- **Factually incorrect**: Making up facts, statistics, or details
- **Not grounded in context**: Providing answers that aren't supported by the provided source material
- **Irrelevant**: Responding to questions with off-topic information

**Why this matters:**
- **RAG Systems**: Ensure responses are grounded in retrieved documents, not fabricated
- **Healthcare**: Prevent dangerous medical misinformation
- **Finance**: Avoid incorrect financial data or statistics
- **Legal**: Prevent fabricated legal precedents or citations
- **Customer Service**: Ensure responses actually answer user questions

EnkryptAI provides two key detectors for anti-hallucination:
- **Adherence**: Checks if LLM answers are supported by provided context
- **Relevancy**: Verifies that responses actually address the user's question

Together, these detectors ensure your AI responses are accurate, grounded, and relevant.

In [1]:
import requests
import os
import json
from dotenv import load_dotenv

# Load environment variables from a .env file, useful for keeping API keys secure
load_dotenv()

# Get your EnkryptAI API key from the environment
ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")

# EnkryptAI Anti-Hallucination endpoints
ADHERENCE_URL = "https://api.enkryptai.com/guardrails/adherence"
RELEVANCY_URL = "https://api.enkryptai.com/guardrails/relevancy"

headers = {
    "apikey": ENKRYPTAI_API_KEY,
    "Content-Type": "application/json"
}

## 1. Adherence (Correctness) Detector

The Adherence detector checks whether an LLM answer is supported by the provided context. This is essential for RAG systems where you need to ensure responses are grounded in retrieved documents.

**How it works:**
1. Extracts atomic facts from the LLM answer
2. Checks each fact against the provided context
3. Returns an adherence score (0-1) and detailed reasoning
4. Provides per-fact scores indicating which parts are supported or unsupported

**Use cases:**
- RAG applications: Ensure responses are grounded in retrieved documents
- Document Q&A: Verify answers reference the source material
- Knowledge bases: Prevent drifting from authoritative content

In [2]:
# Example: LLM answer that contains some correct and some incorrect information
llm_answer = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on making some science stuff or cooking stuff"

# Context from retrieved documents (e.g., from a RAG system)
context = """Indian scientists have made significant contributions to various scientific fields. 
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering. 
Other notable scientists include Srinivasa Ramanujan, a mathematical genius, and A.P.J. Abdul Kalam, 
a key figure in India's aerospace and nuclear programs."""

# Prepare the adherence check request
payload = {
    "llm_answer": llm_answer,
    "context": context
}

# Send the request
response = requests.post(ADHERENCE_URL, json=payload, headers=headers)
result = response.json()

print("LLM Answer:")
print(llm_answer)
print("\n" + "="*80 + "\n")
print("Context:")
print(context)
print("\n" + "="*80 + "\n")
print("Adherence Results:")
print(json.dumps(result, indent=2))

# Extract key information
if "summary" in result and "adherence_score" in result["summary"]:
    score = result["summary"]["adherence_score"]
    print(f"\n{'='*80}")
    print(f"Overall Adherence Score: {score:.2f} ({score*100:.1f}%)")
    
    if score < 0.7:
        print("⚠ Warning: Low adherence detected. The answer may contain unsupported information.")
    else:
        print("✓ Good adherence. The answer is well-supported by the context.")
    
    # Show atomic facts if available
    if "details" in result and "atomic_facts" in result["details"]:
        atomic_facts = result["details"]["atomic_facts"]
        print(f"\nExtracted Atomic Facts: {len(atomic_facts)}")
        for i, fact in enumerate(atomic_facts, 1):
            print(f"  {i}. {fact}")
    
    # Show per-fact adherence scores if available
    if "details" in result and "adherence_list" in result["details"]:
        adherence_list = result["details"]["adherence_list"]
        atomic_facts = result["details"].get("atomic_facts", [])
        print(f"\nPer-Fact Adherence Scores:")
        for i, (fact, score) in enumerate(zip(atomic_facts, adherence_list), 1):
            status = "✓ Supported" if score == 2 else "✗ Unsupported/Incorrect"
            print(f"  {i}. [{status}] {fact}")

LLM Answer:
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on making some science stuff or cooking stuff


Context:
Indian scientists have made significant contributions to various scientific fields. 
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering. 
Other notable scientists include Srinivasa Ramanujan, a mathematical genius, and A.P.J. Abdul Kalam, 
a key figure in India's aerospace and nuclear programs.


Adherence Results:
{
  "summary": {
    "adherence_score": 0.5
  },
  "details": {
    "atomic_facts": [
      "C.V. Raman won the Nobel Prize for Physics in 1930.",
      "C.V. Raman won the Nobel Prize for his work on making some science stuff or cooking stuff."
    ],
    "adherence_list": [
      2,
      0
    ],
    "adherence_response": "{\n    \"chain_of_thought\": [\n        \"TYPE 2: This fact is directly supported by the context, which states that C.V. Raman won the Nobel Prize for Physics in 1930.\",\n        \"TYPE 0:

## 2. Relevancy Detector

The Relevancy detector measures how relevant an LLM's response is to the user's question. This ensures that the AI actually addresses what was asked, rather than providing off-topic or evasive answers.

**How it works:**
- Compares the LLM answer against the original question
- Determines if the response directly addresses the question
- Returns a relevancy score and detailed analysis

**Use cases:**
- Q&A systems: Ensure responses actually answer the question
- Customer service: Detect off-topic or evasive answers
- Quality control: Verify response quality before presenting to users

In [3]:
# Example: Question and LLM answer
question = "What is CV Raman known for?"
llm_answer = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering"

# Prepare the relevancy check request
payload = {
    "question": question,
    "llm_answer": llm_answer
}

# Send the request
response = requests.post(RELEVANCY_URL, json=payload, headers=headers)
result = response.json()

print("Question:")
print(question)
print("\n" + "="*80 + "\n")
print("LLM Answer:")
print(llm_answer)
print("\n" + "="*80 + "\n")
print("Relevancy Results:")
print(json.dumps(result, indent=2))

# Extract key information
if "summary" in result and "relevancy_score" in result["summary"]:
    score = result["summary"]["relevancy_score"]
    print(f"\n{'='*80}")
    print(f"Relevancy Score: {score:.2f} ({score*100:.1f}%)")
    
    if score < 0.7:
        print("⚠ Warning: Low relevancy detected. The answer may not fully address the question.")
    else:
        print("✓ Good relevancy. The answer directly addresses the question.")

Question:
What is CV Raman known for?


LLM Answer:
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering


Relevancy Results:
{
  "summary": {
    "relevancy_score": 1.0
  },
  "details": {
    "atomic_facts": [
      "C.V. Raman won the Nobel Prize for Physics in 1930.",
      "C.V. Raman won the Nobel Prize for his work on light scattering."
    ],
    "relevancy_list": [
      1,
      1
    ],
    "relevancy_response": "{\n    \"chain_of_thought\": [\n        \"RELEVANT: Winning the Nobel Prize is a significant achievement and relevant to what C.V. Raman is known for.\",\n        \"RELEVANT: His work on light scattering is a key contribution and relevant to what he is known for.\"\n    ],\n    \"relevancy\": [\n        1,\n        1\n    ]\n}",
    "relevancy_latency": 1.101151704788208
  }
}


## 3. Using Both Detectors in a RAG System

In a RAG (Retrieval-Augmented Generation) system, you can use both detectors together to ensure high-quality responses:

1. **Retrieve** relevant documents from your knowledge base
2. **Generate** an answer using an LLM with the retrieved context
3. **Check Adherence** to ensure the answer is grounded in the retrieved documents
4. **Check Relevancy** to ensure the answer addresses the user's question

This workflow ensures your RAG system produces accurate, grounded, and relevant responses.

In [7]:
# Simulated RAG workflow
def rag_workflow_with_guardrails(user_question, retrieved_context, llm_answer):
    """
    Simulates a RAG system with anti-hallucination guardrails.
    In a real system, you would:
    1. Retrieve documents based on user_question
    2. Generate llm_answer using the retrieved context
    3. Run both guardrails checks
    """
    print("="*80)
    print("RAG WORKFLOW WITH ANTI-HALLUCINATION GUARDRAILS")
    print("="*80)
    
    print(f"\n📝 User Question: {user_question}")
    print(f"\n📚 Retrieved Context: {retrieved_context[:100]}...")
    print(f"\n🤖 LLM Answer: {llm_answer}")
    
    # Check Adherence
    print("\n" + "="*80)
    print("1. CHECKING ADHERENCE (Is answer grounded in context?)")
    print("="*80)
    
    adherence_payload = {
        "llm_answer": llm_answer,
        "context": retrieved_context
    }
    adherence_response = requests.post(ADHERENCE_URL, json=adherence_payload, headers=headers)
    adherence_result = adherence_response.json()
    
    # Extract score from summary (API returns {"summary": {"adherence_score": ...}})
    adherence_score = adherence_result.get("summary", {}).get("adherence_score", 0)
    print(f"Adherence Score: {adherence_score:.2f} ({adherence_score*100:.1f}%)")
    
    # Check Relevancy
    print("\n" + "="*80)
    print("2. CHECKING RELEVANCY (Does answer address the question?)")
    print("="*80)
    
    relevancy_payload = {
        "question": user_question,
        "llm_answer": llm_answer
    }
    relevancy_response = requests.post(RELEVANCY_URL, json=relevancy_payload, headers=headers)
    relevancy_result = relevancy_response.json()
    
    # Extract score from summary (API returns {"summary": {"relevancy_score": ...}})
    relevancy_score = relevancy_result.get("summary", {}).get("relevancy_score", 0)
    print(f"Relevancy Score: {relevancy_score:.2f} ({relevancy_score*100:.1f}%)")
    
    # Final decision
    print("\n" + "="*80)
    print("3. FINAL ASSESSMENT")
    print("="*80)
    
    if adherence_score >= 0.7 and relevancy_score >= 0.7:
        print("✅ Response approved: High adherence and relevancy")
        return True
    elif adherence_score < 0.7:
        print("❌ Response rejected: Low adherence (answer not grounded in context)")
        return False
    elif relevancy_score < 0.7:
        print("❌ Response rejected: Low relevancy (answer doesn't address the question)")
        return False
    else:
        print("⚠️ Response flagged: Both scores below threshold")
        return False

# Example usage
user_question = "What is CV Raman known for?"

retrieved_context = """Indian scientists have made significant contributions to various scientific fields. 
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering."""
llm_answer = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering"

# Run the workflow
approved = rag_workflow_with_guardrails(user_question, retrieved_context, llm_answer)

RAG WORKFLOW WITH ANTI-HALLUCINATION GUARDRAILS

📝 User Question: What is CV Raman known for?

📚 Retrieved Context: Indian scientists have made significant contributions to various scientific fields. 
C.V. Raman won ...

🤖 LLM Answer: C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering

1. CHECKING ADHERENCE (Is answer grounded in context?)
Adherence Score: 1.00 (100.0%)

2. CHECKING RELEVANCY (Does answer address the question?)
Relevancy Score: 1.00 (100.0%)

3. FINAL ASSESSMENT
✅ Response approved: High adherence and relevancy


## Understanding the Response Format

### Adherence Response Structure

The adherence endpoint returns:
- **`adherence_score`**: Overall score (0-1), where 1.0 = 100% adherence
- **`atomic_facts`**: List of extracted facts from the LLM answer
- **`adherence_list`**: Per-fact scores (2 = supported, 0 = unsupported/incorrect)
- **`adherence_response`**: Chain-of-thought explanation for each fact

**Example:**
```json
{
  "adherence_score": 0.5,
  "atomic_facts": [
    "C.V. Raman won the Nobel Prize for Physics in 1930",
    "for his work on making some science stuff or cooking stuff"
  ],
  "adherence_list": [2, 0],
  "adherence_response": [
    "This fact is supported by the context...",
    "This fact is not supported by the context..."
  ]
}
```

### Relevancy Response Structure

The relevancy endpoint returns:
- **`relevancy_score`**: Score (0-1), where 1.0 = 100% relevant
- Additional metadata about the relevancy assessment

**Example:**
```json
{
  "relevancy_score": 0.95,
  ...
}
```

## Summary

This notebook demonstrated two key anti-hallucination capabilities:

1. **Adherence Detector**: Ensures LLM answers are grounded in provided context
   - Endpoint: `/guardrails/adherence`
   - Input: `llm_answer` and `context`
   - Returns: Adherence score, atomic facts, and per-fact analysis

2. **Relevancy Detector**: Verifies that responses address the user's question
   - Endpoint: `/guardrails/relevancy`
   - Input: `question` and `llm_answer`
   - Returns: Relevancy score and analysis

### Best Practices

- **Always use both detectors in RAG systems**: Check both adherence and relevancy
- **Set appropriate thresholds**: Typically 0.7 (70%) is a good baseline
- **Provide rich context**: Better context leads to more accurate adherence checks
- **Monitor scores over time**: Track adherence and relevancy to improve your RAG system

### Common Use Cases

- **RAG Applications**: Ensure responses are grounded in retrieved documents
- **Document Q&A**: Verify answers reference source material
- **Customer Service**: Ensure responses actually answer questions
- **Healthcare/Legal/Finance**: Critical for fact-critical applications